# Week 12: Neural Networks — Building Blocks to Intelligence

## Learning Objectives

By the end of this notebook you will be able to:
- Prove why stacking linear layers adds no expressive power
- Describe the role and trade-offs of 4 activation functions
- Trace a forward pass through a small network by hand
- Explain backpropagation intuitively (without calculus)
- Apply MLPClassifier to problems that defeat linear models
- State practical architecture design rules of thumb

---

> **The central mystery of this week:**  
> A single linear layer learns a line.  
> Two stacked linear layers also learn... just a line.  
> So how does stacking layers help at all? The answer: **activation functions**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from sklearn.neural_network  import MLPClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.datasets        import make_moons, make_circles
from sklearn.metrics         import accuracy_score
from sklearn.preprocessing   import StandardScaler

np.random.seed(42)
print("Libraries loaded.")

---

## Part 1: The Problem with Stacking Linear Layers

A linear transformation can be written as a matrix multiplication: $h = Wx + b$

If we stack two linear layers:  
$$h_1 = W_1 x + b_1$$
$$y = W_2 h_1 + b_2 = W_2(W_1 x + b_1) + b_2 = (W_2 W_1)\, x + (W_2 b_1 + b_2)$$

The product $W_2 W_1$ is just another matrix — call it $W^*$.  
So two stacked linear layers = one single linear layer. **Depth adds nothing without activation.**

In [ ]:
# ── Prove it with numbers ──────────────────────────────────────────────────
np.random.seed(0)

# Input: batch of 5 data points, 3 features each
X_demo = np.random.randn(5, 3)

# Layer 1: 3 → 4
W1 = np.random.randn(4, 3)
b1 = np.random.randn(4)

# Layer 2: 4 → 2
W2 = np.random.randn(2, 4)
b2 = np.random.randn(2)

# Two-layer forward pass (NO activation)
h1 = X_demo @ W1.T + b1   # shape: (5, 4)
y_two_layers = h1 @ W2.T + b2   # shape: (5, 2)

# Equivalent single-layer: W* = W2 @ W1, b* = W2 @ b1 + b2
W_star = W2 @ W1          # shape: (2, 3)
b_star = W2 @ b1 + b2     # shape: (2,)
y_one_layer = X_demo @ W_star.T + b_star  # shape: (5, 2)

print("=== Two Linear Layers vs One Equivalent Layer ===")
print()
print("Output from 2 stacked layers:")
print(np.round(y_two_layers, 6))
print()
print("Output from equivalent single layer (W2@W1, W2@b1+b2):")
print(np.round(y_one_layer, 6))
print()
print(f"Max absolute difference: {np.abs(y_two_layers - y_one_layer).max():.2e}")
print()
print("CONCLUSION: The outputs are identical (up to floating-point precision).")
print("Two linear layers collapse into one. Depth is meaningless without activation.")

In [ ]:
# ── Visual summary ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

x_vis = np.linspace(-3, 3, 200)

# Without activation: stacking 3 linear functions
f1 = 2.0 * x_vis + 1.0
f2 = -0.5 * f1 + 0.3         # compose
f3 = 1.5 * f2 - 0.8          # compose again — still linear!
f_direct = 1.5 * (-0.5 * (2.0 * x_vis + 1.0) + 0.3) - 0.8   # combined

axes[0].plot(x_vis, f3, 'b-',   lw=3, label='3 composed linear layers')
axes[0].plot(x_vis, f_direct, 'r--', lw=1.5, label='Equivalent single line')
axes[0].set_title("Without Activation: Still Linear", fontsize=11)
axes[0].set_xlabel("x"); axes[0].set_ylabel("output")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
axes[0].text(0, f3[100]+0.5, "Identical!", ha='center', fontsize=11,
             color='darkred', fontweight='bold')

# With ReLU activation: becomes non-linear
relu = lambda z: np.maximum(0, z)
h1_vis = relu(2.0 * x_vis + 1.0)
h2_vis = relu(-0.5 * h1_vis + 1.2)
y_nonlin = 1.5 * h2_vis - 0.8

axes[1].plot(x_vis, y_nonlin, 'g-', lw=3, label='3 layers WITH ReLU')
axes[1].set_title("With ReLU Activation: Truly Non-linear", fontsize=11)
axes[1].set_xlabel("x"); axes[1].set_ylabel("output")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].text(0, y_nonlin[100]+0.15, "Non-linear!", ha='center', fontsize=11,
             color='darkgreen', fontweight='bold')

plt.suptitle("Activation Functions: The Key to Non-linear Learning", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 2: Activation Functions

Activation functions introduce **non-linearity** between layers.  
Without them, a neural network of any depth is just a linear model.

We'll look at 4 functions, their properties, and their trade-offs.

In [ ]:
# ── Define all 4 activation functions and their derivatives ───────────────
x_act = np.linspace(-4, 4, 400)

# 1. ReLU
def relu(x):       return np.maximum(0, x)
def relu_deriv(x): return (x > 0).astype(float)

# 2. Sigmoid
def sigmoid(x):       return 1 / (1 + np.exp(-x))
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

# 3. Tanh
def tanh(x):       return np.tanh(x)
def tanh_deriv(x): return 1 - np.tanh(x)**2

# 4. Leaky ReLU
def leaky_relu(x):       return np.where(x >= 0, x, 0.1 * x)
def leaky_relu_deriv(x): return np.where(x >= 0, 1.0, 0.1)

# ── Apply ReLU manually to a sample vector ────────────────────────────────
sample = np.array([-2.0, -1.0, 0.0, 0.5, 1.0, 2.0, 3.0])
print("=== ReLU Applied to Sample Vector ===")
print(f"Input:   {sample}")
print(f"ReLU:    {relu(sample)}")
print(f"Sigmoid: {np.round(sigmoid(sample), 3)}")
print(f"Tanh:    {np.round(tanh(sample), 3)}")
print(f"LeakyRL: {leaky_relu(sample)}")
print()
print("ReLU dead neuron: inputs -2, -1 → output 0 (no gradient flows)")
print("Leaky ReLU fix: inputs -2, -1 → -0.2, -0.1 (small gradient still flows)")

In [ ]:
# ── Plot all 4 functions with their derivatives ───────────────────────────
funcs = [
    (relu,       relu_deriv,       'ReLU',        '#e74c3c',
     'max(0,x)\n• Simple, fast\n• Dead neuron problem\n• Default for hidden layers'),
    (sigmoid,    sigmoid_deriv,    'Sigmoid',     '#3498db',
     '1/(1+e⁻ˣ)\n• Output ∈ (0,1)\n• Saturates at extremes\n• Use for binary output'),
    (tanh,       tanh_deriv,       'Tanh',        '#2ecc71',
     '(eˣ-e⁻ˣ)/(eˣ+e⁻ˣ)\n• Output ∈ (-1,1)\n• Zero-centered\n• Better than sigmoid'),
    (leaky_relu, leaky_relu_deriv, 'Leaky ReLU',  '#9b59b6',
     'max(0.1x, x)\n• Fixes dead neuron\n• Slightly slower\n• Alternative to ReLU'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))

for col, (fn, dfn, name, color, desc) in enumerate(funcs):
    # Top row: the function
    ax = axes[0, col]
    ax.plot(x_act, fn(x_act), color=color, lw=2.5)
    ax.axhline(0, color='black', lw=0.8, alpha=0.5)
    ax.axvline(0, color='black', lw=0.8, alpha=0.5)
    ax.set_title(f"{name}\n{desc}", fontsize=9)
    ax.set_xlabel("z")
    ax.set_ylabel("f(z)")
    ax.grid(alpha=0.3)
    ax.set_xlim(-4, 4)

    # Bottom row: the derivative
    ax2 = axes[1, col]
    ax2.plot(x_act, dfn(x_act), color=color, lw=2.5, linestyle='--')
    ax2.axhline(0, color='black', lw=0.8, alpha=0.5)
    ax2.set_title(f"d/dz {name}", fontsize=10)
    ax2.set_xlabel("z")
    ax2.set_ylabel("f'(z)")
    ax2.grid(alpha=0.3)
    ax2.set_xlim(-4, 4)

plt.suptitle("Activation Functions and Their Derivatives\n(Derivatives show gradient flow)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observation from derivatives:")
print("  Sigmoid/Tanh: derivative approaches 0 for large |x| → vanishing gradient")
print("  ReLU: derivative is 1 for x>0, 0 for x<0 → clean gradient flow or dead neuron")
print("  Leaky ReLU: derivative is 1 for x>0, 0.1 for x<0 → always some gradient")

---

## Part 3: Hidden Layers as Feature Learners

The key insight: **hidden units learn combinations of the inputs**, just like engineered features — but automatically.

Let's trace a complete forward pass manually for a loan approval network:

```
Inputs:  age (normalized), income (normalized)
Layer 1: 2 hidden units with ReLU
Output:  1 unit with sigmoid → P(loan approved)
```

In [ ]:
# ── Manual forward pass ────────────────────────────────────────────────────

# Input: normalized age=0.6 (≈35yo), normalized income=0.8 (good income)
x = np.array([0.6, 0.8])

print("=" * 55)
print("INPUT")
print("=" * 55)
print(f"  x[0] = {x[0]}  (normalized age)")
print(f"  x[1] = {x[1]}  (normalized income)")

# Layer 1 weights: 2 inputs → 2 hidden units
# W1 shape: (n_hidden, n_inputs) = (2, 2)
W1 = np.array([[1.2, -0.5],
               [0.3,  1.1]])
b1 = np.array([-0.4, 0.1])

print()
print("=" * 55)
print("LAYER 1: Linear combination (z = W1·x + b1)")
print("=" * 55)
z1 = W1 @ x + b1
print(f"  z1[0] = {W1[0,0]}*{x[0]} + {W1[0,1]}*{x[1]} + ({b1[0]}) = {z1[0]:.4f}")
print(f"  z1[1] = {W1[1,0]}*{x[0]} + {W1[1,1]}*{x[1]} + ({b1[1]}) = {z1[1]:.4f}")

print()
print("=" * 55)
print("LAYER 1: ReLU Activation (h1 = ReLU(z1))")
print("=" * 55)
h1 = relu(z1)
print(f"  h1[0] = ReLU({z1[0]:.4f}) = {h1[0]:.4f}")
print(f"  h1[1] = ReLU({z1[1]:.4f}) = {h1[1]:.4f}")
print(f"  h1 represents: learned combinations of age and income")

# Layer 2 weights: 2 hidden → 1 output
W2 = np.array([[0.9, 1.3]])
b2 = np.array([-0.6])

print()
print("=" * 55)
print("LAYER 2: Linear combination (z2 = W2·h1 + b2)")
print("=" * 55)
z2 = W2 @ h1 + b2
print(f"  z2 = {W2[0,0]}*{h1[0]:.4f} + {W2[0,1]}*{h1[1]:.4f} + ({b2[0]}) = {z2[0]:.4f}")

print()
print("=" * 55)
print("OUTPUT: Sigmoid → P(approved)")
print("=" * 55)
y_hat = sigmoid(z2)
print(f"  ŷ = sigmoid({z2[0]:.4f}) = {y_hat[0]:.4f}")
print(f"  → P(loan approved) = {y_hat[0]*100:.1f}%")
print(f"  → Decision: {'APPROVE' if y_hat[0] > 0.5 else 'DENY'}")

```
Forward pass diagram:

  x[0]=0.6 ──W1[0,0]=1.2──┐
                            ├──→ z1[0] ──ReLU──→ h1[0] ──W2[0,0]=0.9──┐
  x[1]=0.8 ──W1[0,1]=-0.5─┘                                            ├──→ z2 ──sigmoid──→ ŷ
                                                                         │
  x[0]=0.6 ──W1[1,0]=0.3──┐                                            │
                            ├──→ z1[1] ──ReLU──→ h1[1] ──W2[0,1]=1.3──┘
  x[1]=0.8 ──W1[1,1]=1.1──┘
```

Each hidden unit $h_i$ is a **learned combination** of the inputs — the network discovers its own features.

---

## Part 4: Three Problems — Linear Model vs Neural Network

We'll attack three problems that defeat linear models but fall easily to a shallow neural network.

In [ ]:
# ── Utility: plot decision boundary for 2D classifier ─────────────────────
def plot_decision_boundary(ax, model, X, y, title, scaler=None):
    """Plots a filled decision boundary contour for a 2D classifier."""
    x_min = X[:, 0].min() - 0.5
    x_max = X[:, 0].max() + 0.5
    y_min = X[:, 1].min() - 0.5
    y_max = X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                         np.linspace(y_min, y_max, 250))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if scaler:
        grid = scaler.transform(grid)
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu',
               edgecolors='k', linewidths=0.4, s=30, zorder=3)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.2)
    return ax

print("Utility function defined.")

In [ ]:
# ── Problem 1: XOR ────────────────────────────────────────────────────────
np.random.seed(3)
n = 200
x1_xor = np.random.uniform(-2, 2, n)
x2_xor = np.random.uniform(-2, 2, n)
y_xor  = ((x1_xor * x2_xor) > 0).astype(int)
# small noise
flip = np.random.rand(n) < 0.05
y_xor[flip] = 1 - y_xor[flip]
X_xor = np.column_stack([x1_xor, x2_xor])

# ── Problem 2: Concentric circles ─────────────────────────────────────────
X_circ, y_circ = make_circles(n_samples=300, noise=0.12, factor=0.4, random_state=5)

# ── Problem 3: Moons (sklearn.datasets.make_moons) ────────────────────────
X_moon, y_moon = make_moons(n_samples=400, noise=0.22, random_state=7)

print("Datasets created:")
print(f"  XOR:     {X_xor.shape[0]} samples")
print(f"  Circles: {X_circ.shape[0]} samples")
print(f"  Moons:   {X_moon.shape[0]} samples")

In [ ]:
# ── Train linear and NN models on each problem ────────────────────────────
scaler = StandardScaler()

def train_and_eval(X, y, dataset_name):
    """Train logistic regression and MLP, return models + accuracies."""
    X_sc = scaler.fit_transform(X)

    lr_model = LogisticRegression(max_iter=500).fit(X_sc, y)
    nn_model = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation='relu',
        max_iter=1000,
        random_state=0
    ).fit(X_sc, y)

    lr_acc = accuracy_score(y, lr_model.predict(X_sc))
    nn_acc = accuracy_score(y, nn_model.predict(X_sc))

    print(f"{dataset_name:12s} — Logistic Regression: {lr_acc:.3f}  |  Neural Network: {nn_acc:.3f}")
    return lr_model, nn_model, X_sc, scaler

print("=" * 65)
print(f"{'Dataset':12s}   {'Linear Accuracy':20s}   {'NN Accuracy':15s}")
print("=" * 65)

# XOR
sc_xor  = StandardScaler()
X_xor_sc = sc_xor.fit_transform(X_xor)
lr_xor  = LogisticRegression(max_iter=500).fit(X_xor_sc, y_xor)
nn_xor  = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', max_iter=1000, random_state=0).fit(X_xor_sc, y_xor)
print(f"{'XOR':12s} — LR: {accuracy_score(y_xor, lr_xor.predict(X_xor_sc)):.3f}  |  NN: {accuracy_score(y_xor, nn_xor.predict(X_xor_sc)):.3f}")

# Circles
sc_circ  = StandardScaler()
X_circ_sc = sc_circ.fit_transform(X_circ)
lr_circ = LogisticRegression(max_iter=500).fit(X_circ_sc, y_circ)
nn_circ = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', max_iter=1000, random_state=0).fit(X_circ_sc, y_circ)
print(f"{'Circles':12s} — LR: {accuracy_score(y_circ, lr_circ.predict(X_circ_sc)):.3f}  |  NN: {accuracy_score(y_circ, nn_circ.predict(X_circ_sc)):.3f}")

# Moons
sc_moon  = StandardScaler()
X_moon_sc = sc_moon.fit_transform(X_moon)
lr_moon = LogisticRegression(max_iter=500).fit(X_moon_sc, y_moon)
nn_moon = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=1000, random_state=0).fit(X_moon_sc, y_moon)
print(f"{'Moons':12s} — LR: {accuracy_score(y_moon, lr_moon.predict(X_moon_sc)):.3f}  |  NN: {accuracy_score(y_moon, nn_moon.predict(X_moon_sc)):.3f}")

In [ ]:
# ── Plot decision boundaries for all 3 problems ───────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 14))

problems = [
    (X_xor,  y_xor,  lr_xor,  nn_xor,  sc_xor,  'XOR Problem'),
    (X_circ, y_circ, lr_circ, nn_circ, sc_circ, 'Concentric Circles'),
    (X_moon, y_moon, lr_moon, nn_moon, sc_moon, 'Moons (make_moons)'),
]

for row, (X, y, lr_m, nn_m, sc, name) in enumerate(problems):
    lr_acc = accuracy_score(y, lr_m.predict(sc.transform(X)))
    nn_acc = accuracy_score(y, nn_m.predict(sc.transform(X)))

    plot_decision_boundary(
        axes[row, 0], lr_m, X, y,
        f"Linear Model — {name}\nAccuracy: {lr_acc:.3f}",
        scaler=sc
    )
    plot_decision_boundary(
        axes[row, 1], nn_m, X, y,
        f"Neural Network — {name}\nAccuracy: {nn_acc:.3f}",
        scaler=sc
    )
    for col in range(2):
        axes[row, col].set_xlabel("Feature 1")
        axes[row, col].set_ylabel("Feature 2")

plt.suptitle("Linear Model vs Neural Network Decision Boundaries",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 5: Backpropagation — Intuition Without the Calculus

### The core idea

Once we compute the prediction error, we need to update weights.  
But which weight is responsible for how much of the error?

Backpropagation answers this by **tracing error back through the network**, layer by layer.

> **Analogy:** Imagine a factory assembly line where a defective part slips through.  
> You trace back: Inspector 3 → Inspector 2 → Machine 1.  
> Each station gets blamed proportionally to how much it influenced the final product.

The **chain rule** of calculus formalizes this: if A caused 30% of B's activation,  
and B caused 40% of the error, then A is responsible for 30% × 40% = 12% of the error.

In [ ]:
# ── Trace blame through a 3-node network ──────────────────────────────────
print("=" * 60)
print("BACKPROPAGATION WORKED EXAMPLE")
print("=" * 60)
print()
print("Network: x → h (hidden, ReLU) → ŷ (output, linear)")
print()

# Forward pass
x_bp = 2.0
w1_bp = 0.5    # weight from x to hidden
b1_bp = -0.3
w2_bp = 1.2    # weight from hidden to output
b2_bp = 0.1
y_true_bp = 1.5

z1_bp = w1_bp * x_bp + b1_bp
h_bp  = relu(np.array([z1_bp]))[0]    # ReLU
y_pred_bp = w2_bp * h_bp + b2_bp
loss_bp = (y_pred_bp - y_true_bp)**2  # MSE

print("--- FORWARD PASS ---")
print(f"  x        = {x_bp}")
print(f"  z1       = w1*x + b1 = {w1_bp}*{x_bp} + {b1_bp} = {z1_bp:.2f}")
print(f"  h        = ReLU({z1_bp:.2f}) = {h_bp:.2f}")
print(f"  ŷ        = w2*h + b2 = {w2_bp}*{h_bp:.2f} + {b2_bp} = {y_pred_bp:.2f}")
print(f"  y_true   = {y_true_bp}")
print(f"  Error    = ŷ - y = {y_pred_bp - y_true_bp:.2f}")
print(f"  Loss     = (ŷ-y)² = {loss_bp:.4f}")
print()

# Backward pass
d_loss_d_ypred = 2 * (y_pred_bp - y_true_bp)   # dL/dŷ
d_ypred_d_w2   = h_bp                           # dŷ/dw2
d_ypred_d_h    = w2_bp                          # dŷ/dh
d_h_d_z1       = 1.0 if z1_bp > 0 else 0.0     # dh/dz1 (ReLU derivative)
d_z1_d_w1      = x_bp                           # dz1/dw1

# Chain rule
d_loss_d_w2 = d_loss_d_ypred * d_ypred_d_w2
d_loss_d_w1 = d_loss_d_ypred * d_ypred_d_h * d_h_d_z1 * d_z1_d_w1

print("--- BACKWARD PASS (Chain Rule) ---")
print(f"  dL/dŷ    = 2*(ŷ-y) = {d_loss_d_ypred:.4f}  (total error signal)")
print()
print(f"  For w2:  dL/dw2 = dL/dŷ × dŷ/dw2")
print(f"           = {d_loss_d_ypred:.4f} × {d_ypred_d_w2:.4f} = {d_loss_d_w2:.4f}")
print()
print(f"  For w1:  dL/dw1 = dL/dŷ × dŷ/dh × dh/dz1 × dz1/dw1")
print(f"           = {d_loss_d_ypred:.4f} × {d_ypred_d_h:.4f} × {d_h_d_z1:.4f} × {d_z1_d_w1:.4f}")
print(f"           = {d_loss_d_w1:.4f}")
print()

lr_bp = 0.01
print("--- WEIGHT UPDATE (lr=0.01) ---")
print(f"  w2_new = {w2_bp} - {lr_bp} × {d_loss_d_w2:.4f} = {w2_bp - lr_bp * d_loss_d_w2:.4f}")
print(f"  w1_new = {w1_bp} - {lr_bp} × {d_loss_d_w1:.4f} = {w1_bp - lr_bp * d_loss_d_w1:.4f}")
print()
print("Fraction of blame reaching w1 vs w2:")
print(f"  |grad_w2| = {abs(d_loss_d_w2):.4f}")
print(f"  |grad_w1| = {abs(d_loss_d_w1):.4f}")
print(f"  → w1 receives {abs(d_loss_d_w1)/abs(d_loss_d_w2)*100:.0f}% as much blame as w2")
print("    (because w1 is one step further from the loss)")

In [ ]:
# ── Visual: how blame propagates backward ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis('off')

# Nodes: positions
nodes = {
    'x':     (0.1, 0.5),
    'h':     (0.45, 0.5),
    'y_hat': (0.80, 0.5),
    'loss':  (1.05, 0.5),
}

# Draw nodes
for name, (px, py) in nodes.items():
    color = '#f39c12' if name == 'loss' else '#3498db'
    circle = plt.Circle((px, py), 0.06, color=color, zorder=3)
    ax.add_patch(circle)
    label = {'x': 'x=2.0', 'h': f'h={h_bp:.2f}', 'y_hat': f'ŷ={y_pred_bp:.2f}', 'loss': f'L={loss_bp:.3f}'}[name]
    ax.text(px, py, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white', zorder=4)

# Forward arrows
arrow_kw = dict(arrowstyle='->', color='steelblue', lw=2)
for (n1, n2), label in [
    (('x', 'h'),     f'w1={w1_bp}'),
    (('h', 'y_hat'), f'w2={w2_bp}'),
    (('y_hat','loss'), 'MSE'),
]:
    x1, y1 = nodes[n1][0]+0.065, nodes[n1][1]
    x2, y2 = nodes[n2][0]-0.065, nodes[n2][1]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=2))
    ax.text((x1+x2)/2, y1+0.10, label, ha='center', fontsize=9, color='steelblue')

# Backward arrows (dashed, red)
for (n1, n2), label in [
    (('loss', 'y_hat'), f"dL/dŷ={d_loss_d_ypred:.2f}"),
    (('y_hat', 'h'),    f"×w2={w2_bp}"),
    (('h', 'x'),        f"×ReLU'×x\n={d_loss_d_w1:.3f}"),
]:
    x1, y1 = nodes[n1][0]-0.065, nodes[n1][1]-0.03
    x2, y2 = nodes[n2][0]+0.065, nodes[n2][1]-0.03
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='crimson', lw=2, linestyle='dashed'))
    ax.text((x1+x2)/2, y1-0.12, label, ha='center', fontsize=8.5, color='crimson')

# Legend
ax.text(0.5, 0.92, 'FORWARD PASS (blue)', ha='center', fontsize=11, color='steelblue',
        transform=ax.transAxes)
ax.text(0.5, 0.05, 'BACKWARD PASS — blame propagates right to left (red dashed)',
        ha='center', fontsize=11, color='crimson', transform=ax.transAxes)
ax.set_xlim(-0.05, 1.2)
ax.set_ylim(0.1, 0.9)
ax.set_title("Backpropagation: Tracing the Error Back Through the Network",
             fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()

### The "rumor source" analogy

> Imagine a game of telephone where a rumor gets distorted.  
> The final message is wrong. To find out who started the distortion, you trace back:  
> - Person C heard it from Person B  
> - Person B got 70% of their message from Person A  
> - So Person A gets 70% of the blame  
> 
> Chain rule = tracing proportional blame through a chain of people.

---

## Part 6: Architecture Design — Rules of Thumb

There is no single formula for the perfect neural network architecture, but these rules guide a solid first attempt.

In [ ]:
# ── Demonstrate how architecture choices affect accuracy ──────────────────
X_demo_arch, y_demo_arch = make_moons(n_samples=600, noise=0.25, random_state=42)
X_sc_arch = StandardScaler().fit_transform(X_demo_arch)
X_tr_arch, X_te_arch, y_tr_arch, y_te_arch = (
    X_sc_arch[:450], X_sc_arch[450:], y_demo_arch[:450], y_demo_arch[450:]
)

architectures = [
    ((4,),       'Tiny: (4,)'),
    ((16,),      'Shallow wide: (16,)'),
    ((8, 4),     'Narrow deep: (8,4)'),
    ((32, 16),   'Standard: (32,16)'),
    ((64, 32, 16),'Deeper: (64,32,16)'),
    ((128, 64, 32, 16), 'Very deep: (128,64,32,16)'),
]

print(f"{'Architecture':30s}  {'Train Acc':10s}  {'Test Acc':10s}  {'Notes':20s}")
print("-" * 75)

for hidden_sizes, label in architectures:
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_sizes,
        activation='relu',
        max_iter=1000,
        random_state=0
    ).fit(X_tr_arch, y_tr_arch)
    tr_acc = accuracy_score(y_tr_arch, mlp.predict(X_tr_arch))
    te_acc = accuracy_score(y_te_arch, mlp.predict(X_te_arch))
    gap = tr_acc - te_acc
    note = 'underfitting' if te_acc < 0.87 else ('slight overfit' if gap > 0.04 else 'good')
    print(f"{label:30s}  {tr_acc:.3f}      {te_acc:.3f}      {note}")

In [ ]:
# ── Architecture diagram as a matplotlib figure ───────────────────────────
def draw_network(ax, layer_sizes, title):
    """Draw a simple neural network diagram."""
    ax.axis('off')
    ax.set_xlim(-0.5, len(layer_sizes) - 0.5)
    max_nodes = max(layer_sizes)
    ax.set_ylim(-0.5, max_nodes + 0.5)
    ax.set_title(title, fontsize=10)

    layer_colors = ['#3498db', '#e67e22', '#e67e22', '#2ecc71']
    node_positions = []

    for l_idx, n_nodes in enumerate(layer_sizes):
        color = layer_colors[min(l_idx, len(layer_colors)-1)]
        y_offset = (max_nodes - n_nodes) / 2  # center vertically
        positions = []
        for n_idx in range(n_nodes):
            y_pos = y_offset + n_idx
            circle = plt.Circle((l_idx, y_pos), 0.18, color=color, zorder=3, alpha=0.9)
            ax.add_patch(circle)
            positions.append((l_idx, y_pos))
        node_positions.append(positions)

    # Draw edges
    for l in range(len(layer_sizes) - 1):
        for src in node_positions[l]:
            for dst in node_positions[l+1]:
                ax.plot([src[0]+0.18, dst[0]-0.18], [src[1], dst[1]],
                        color='gray', lw=0.5, alpha=0.4, zorder=1)

    # Layer labels
    layer_names = ['Input', 'Hidden 1', 'Hidden 2', 'Output']
    for l_idx, name in enumerate(layer_names[:len(layer_sizes)]):
        ax.text(l_idx, -0.35, name, ha='center', fontsize=8, color='gray')
        ax.text(l_idx, -0.05 + max_nodes, f"n={layer_sizes[l_idx]}",
                ha='center', fontsize=7.5, color='dimgray')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Example 1: Binary classification (2 inputs, 2 hidden layers, 1 output)
draw_network(axes[0], [2, 4, 3, 1],
             "Binary Classification\n[2 → 4 → 3 → 1]\nOutput: sigmoid")
axes[0].text(3, -0.65, 'sigmoid', ha='center', fontsize=9, color='#2ecc71', fontweight='bold')

# Example 2: Multi-class (5 inputs, 3 hidden layers, 3 classes)
draw_network(axes[1], [5, 6, 4, 3],
             "Multi-class (3 classes)\n[5 → 6 → 4 → 3]\nOutput: softmax")
axes[1].text(3, -0.65, 'softmax', ha='center', fontsize=9, color='#2ecc71', fontweight='bold')

plt.suptitle("Neural Network Architecture Diagrams", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Architecture Design Rules of Thumb

| Decision | Rule of Thumb |
|----------|---------------|
| **Starting point** | 1-2 hidden layers, 32-64 units each |
| **More data available** | Can afford deeper (more layers) and wider (more units) networks |
| **More input features** | Larger first hidden layer (≥ 2× number of features) |
| **Regression output** | Linear activation on output neuron (no squashing) |
| **Binary classification** | Sigmoid on single output neuron → P(class=1) |
| **Multi-class** | Softmax on k output neurons → P(each class) |
| **Hidden layer activation** | ReLU (default), Leaky ReLU if dead neurons are a problem |
| **Overfitting observed** | Add dropout, reduce units/layers, get more data |
| **Underfitting observed** | Add layers, add units, train longer |
| **Normalization** | Always normalize inputs before feeding to a neural network |

---

## Complete Summary

| Concept | Key Point |
|---------|----------|
| **Linear stacking** | Two linear layers = one linear layer. Depth without activation adds nothing. |
| **ReLU** | Fast, default for hidden layers. Dead neuron problem at negative inputs. |
| **Sigmoid** | Output ∈ (0,1), use for binary output. Saturates → vanishing gradients. |
| **Tanh** | Output ∈ (-1,1), zero-centered, better than sigmoid for hidden layers. |
| **Leaky ReLU** | Fixes dead neurons; use when ReLU causes problems. |
| **Hidden layers** | Learn feature combinations automatically — like polynomial features but adaptive. |
| **Forward pass** | Compute predictions layer by layer (linear → activation → linear → ...). |
| **Backpropagation** | Chain rule traces error back — each weight gets proportional blame. |
| **XOR / Circles / Moons** | Linear: ~50–85%. Neural network: ~95–99%. Non-linear boundaries are the key. |
| **Design** | Start simple: 1-2 layers, 32-64 units. Match output activation to task type. |

---

**You now have the complete picture from scalar gradient descent (Week 5) all the way to multi-layer neural networks.** The same core principle applies at every level: compute error, attribute blame, update weights.